# 경구약제 Object Detection 실험

다양한 모델 비교 실험 노트북

**데이터 소스**
- AI Hub 경구약제 조합 데이터 (TS_01~08, TS_02 제외)
- 기존 Kaggle 대회 데이터 (`data/pill_yolo/`, 56클래스)

## 1. 데이터 준비

두 가지 데이터 소스 중 하나를 선택:
- **A. AI Hub 데이터**: 대량 데이터, 서브셋(tiny/small/medium/full) 지원
- **B. 기존 Kaggle 데이터**: `data/pill_yolo/`에 이미 준비된 56클래스 데이터

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parents[1]
sys.path.insert(0, str(PROJECT_ROOT))


In [2]:
from src.joelchoi.data.aihub_converter import convert_aihub_to_coco

# from src.joelchoi.data.yolo_to_coco import load_yolo_as_coco
from src.joelchoi.data.subset import create_subset  # , save_split
from src.joelchoi.data.coco_to_yolo import prepare_yolo_dataset

# from src.joelchoi.data.dataset import (
#     PillDetectionDataset,
#     collate_fn,
#     get_train_augment,
# )
# from src.joelchoi.models.builder import build_model
# from src.joelchoi.models.yolo_wrapper import train_yolo, evaluate_yolo
from src.joelchoi.train import train_torchvision
from src.joelchoi.train_yolo import run_yolo_experiment
from src.joelchoi.utils import load_config  # , set_seed, get_device

CONFIGS_DIR = PROJECT_ROOT / "configs" / "joelchoi"
EXPERIMENTS_DIR = PROJECT_ROOT / "experiments" / "joelchoi"
DATA_DIR = PROJECT_ROOT / "data" / "joelchoi"
KAGGLE_YOLO_DIR = PROJECT_ROOT / "data" / "pill_yolo"

In [3]:
# ============================
# A. AI Hub 데이터 사용 시
# ============================
# combo_nums: 사용할 조합 번호 (2번은 Kaggle 테스트셋이므로 자동 차단)
coco = convert_aihub_to_coco(combo_nums=[1, 3, 4, 5, 6])

변환 중: TS_01/TL_01 ...
  → 이미지 1188장, 어노테이션 4738개
변환 중: TS_03/TL_03 ...
  → 이미지 1491장, 어노테이션 5676개
변환 중: TS_04/TL_04 ...
경고: ._K-005094-005886-012778-037777_0_2_0_2_90_000_200.json 파일을 읽을 수 없어 건너뜁니다.
  → 이미지 1392장, 어노테이션 5023개
변환 중: TS_05/TL_05 ...
  → 이미지 462장, 어노테이션 1845개
변환 중: TS_06/TL_06 ...
경고: K-012247-016551-033009-044199_0_2_0_2_90_000_200.json 파일을 읽을 수 없어 건너뜁니다.
  → 이미지 1122장, 어노테이션 4466개

총 이미지: 5655, 어노테이션: 21748, 클래스: 92


In [4]:
# 서브셋 생성: tiny(100장), small(500장), medium(2000장), full(전체)
TIER = "small"  # 변경하여 데이터 규모 조절
DATA_SOURCE = "aihub"

train_coco, val_coco = create_subset(coco, tier=TIER, seed=42)

[small] train: 400장, val: 100장


### B. 기존 Kaggle 데이터 사용 시 (위 A 대신 이 셀 실행)

In [5]:
# ============================
# B. 기존 Kaggle 대회 데이터 사용 시 (A 대신 이 셀 실행)
# ============================
# data/pill_yolo/ 의 YOLO 포맷 데이터를 COCO로 변환하여 torchvision 모델에서도 사용 가능
# train_coco = load_yolo_as_coco(KAGGLE_YOLO_DIR, split="train")
# val_coco = load_yolo_as_coco(KAGGLE_YOLO_DIR, split="val")
# TIER = "kaggle"
# DATA_SOURCE = "kaggle"

## 2. YOLO 모델 실험

In [6]:
# YOLO용 데이터셋 준비
if DATA_SOURCE == "kaggle":
    # 기존 Kaggle 데이터는 이미 YOLO 포맷 → 바로 사용
    yaml_path = KAGGLE_YOLO_DIR / "data.yaml"
    print(f"기존 Kaggle YOLO 데이터 사용: {yaml_path}")
else:
    # AI Hub 데이터 → YOLO 포맷 변환 (심볼릭 링크로 디스크 절약)
    yolo_dir = DATA_DIR / f"yolo_{TIER}"
    yaml_path = prepare_yolo_dataset(train_coco, val_coco, yolo_dir, symlink=True)

YOLO yaml 저장: /Users/joelchoi/study/projects/project1_3team/data/joelchoi/yolo_small/data.yaml
YOLO 데이터셋 준비 완료: train 400장, val 100장


In [15]:
# YOLO 실험 config 로드 & 학습
yolo_config = load_config(CONFIGS_DIR / "experiment" / "exp001_yolo11n_tiny.yaml")

# 필요시 config 오버라이드
# yolo_config["training"]["epochs"] = 10
# yolo_config["model"]["name"] = "yolo11s"
yolo_config["data"]["subset"] = "small"

yolo_metrics = run_yolo_experiment(yolo_config, yaml_path, EXPERIMENTS_DIR)
print(yolo_metrics)

New https://pypi.org/project/ultralytics/8.4.82 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.80 🚀 Python-3.14.6 torch-2.12.1 CPU (Apple M2 Max)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/Users/joelchoi/study/projects/project1_3team/data/joelchoi/yolo_small/data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=20, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.93

## 3. torchvision 모델 실험 (SSD / Faster R-CNN / RetinaNet / FCOS)

In [7]:
# SSD300 실험
ssd_config = load_config(CONFIGS_DIR / "experiment" / "exp002_ssd300_tiny.yaml")
ssd_metrics = train_torchvision(ssd_config, train_coco, val_coco, EXPERIMENTS_DIR)
print(ssd_metrics)

Device: mps


/Users/joelchoi/study/projects/project1_3team/.venv/lib/python3.14/site-packages/torchmetrics/utilities/prints.py:43: UserWarning: Encountered more than 100 detections in a single image. This means that certain detections with the lowest scores will be ignored, that may have an undesirable impact on performance. Please consider adjusting the `max_detection_threshold` to suit your use case. To disable this warning, set attribute class `warn_on_many_detections=False`, after initializing the metric.
  warnings.warn(*args, **kwargs)


Epoch   1/20 | loss: 9.7644 | mAP@50: 0.3667 | mAP@50:95: 0.2776 | 38s
Epoch   2/20 | loss: 3.5920 | mAP@50: 0.6742 | mAP@50:95: 0.4710 | 37s
Epoch   3/20 | loss: 2.6361 | mAP@50: 0.8271 | mAP@50:95: 0.5159 | 37s
Epoch   4/20 | loss: 2.1844 | mAP@50: 0.8908 | mAP@50:95: 0.5912 | 37s
Epoch   5/20 | loss: 1.7974 | mAP@50: 0.8872 | mAP@50:95: 0.6375 | 37s
Epoch   6/20 | loss: 1.6020 | mAP@50: 0.9313 | mAP@50:95: 0.7056 | 37s
Epoch   7/20 | loss: 1.4062 | mAP@50: 0.9407 | mAP@50:95: 0.6990 | 37s
Epoch   8/20 | loss: 1.3092 | mAP@50: 0.9326 | mAP@50:95: 0.6832 | 37s
Epoch   9/20 | loss: 1.1386 | mAP@50: 0.9624 | mAP@50:95: 0.7428 | 37s
Epoch  10/20 | loss: 0.9856 | mAP@50: 0.9702 | mAP@50:95: 0.8052 | 37s
Epoch  11/20 | loss: 0.8974 | mAP@50: 0.9647 | mAP@50:95: 0.8008 | 37s
Epoch  12/20 | loss: 0.8062 | mAP@50: 0.9665 | mAP@50:95: 0.8257 | 37s
Epoch  13/20 | loss: 0.6950 | mAP@50: 0.9679 | mAP@50:95: 0.8427 | 37s
Epoch  14/20 | loss: 0.6543 | mAP@50: 0.9687 | mAP@50:95: 0.8389 | 37s
Epoch 

In [9]:
# Faster R-CNN 실험
frcnn_config = load_config(CONFIGS_DIR / "experiment" / "exp003_fasterrcnn_tiny.yaml")
frcnn_metrics = train_torchvision(frcnn_config, train_coco, val_coco, EXPERIMENTS_DIR)
print(frcnn_metrics)

Device: mps
Epoch   1/20 | loss: 5.6237 | mAP@50: 0.0000 | mAP@50:95: 0.0000 | 681s
Epoch   2/20 | loss: 20.4538 | mAP@50: 0.0000 | mAP@50:95: 0.0000 | 5328s
Epoch   3/20 | loss: 18.7230 | mAP@50: 0.0000 | mAP@50:95: 0.0000 | 10804s


KeyboardInterrupt: 

In [ ]:
# RetinaNet 실험
retina_config = load_config(CONFIGS_DIR / "experiment" / "exp004_retinanet_tiny.yaml")
retina_metrics = train_torchvision(retina_config, train_coco, val_coco, EXPERIMENTS_DIR)
print(retina_metrics)

In [ ]:
# FCOS 실험
fcos_config = load_config(CONFIGS_DIR / "experiment" / "exp005_fcos_tiny.yaml")
fcos_metrics = train_torchvision(fcos_config, train_coco, val_coco, EXPERIMENTS_DIR)
print(fcos_metrics)

## 4. 결과 비교

In [ ]:
import json
import pandas as pd

results = []
for exp_dir in sorted(EXPERIMENTS_DIR.iterdir()):
    metrics_file = exp_dir / "metrics.json"
    if metrics_file.exists():
        m = json.load(open(metrics_file))
        results.append({
            "experiment": exp_dir.name,
            "model": m.get("model", "unknown"),
            "mAP@75": m.get("map75") or m.get("final_map75", 0),
            "mAP@75:95": m.get("map75_95") or m.get("final_map75_95", 0),
            "epochs": m.get("epochs", 0),
        })

df = pd.DataFrame(results)
df = df.sort_values("mAP@75:95", ascending=False)
df

In [ ]:
import matplotlib.pyplot as plt

if not df.empty:
    fig, ax = plt.subplots(figsize=(10, 5))
    x = range(len(df))
    width = 0.25
    ax.bar(x, df["mAP@75"], width, label="mAP@75")
    ax.bar([i + width for i in x], df["mAP@75:95"], width, label="mAP@75:95")
    ax.set_xticks(x)
    ax.set_xticklabels(df["model"], rotation=45, ha="right")
    ax.set_ylabel("mAP")
    ax.set_title(f"모델별 성능 비교 (tier: {TIER})")
    ax.legend()
    plt.tight_layout()
    plt.show()

## 5. 커스텀 실험

config를 직접 수정하거나 새로 만들어서 실험할 수 있습니다.

In [ ]:
# 예시: YOLO11s + small 데이터셋 + 20 epoch
# custom_config = {
#     "experiment": {"name": "exp010_yolo11s_small", "seed": 42},
#     "data": {"subset": "small", "test_size": 0.2, "img_size": 640},
#     "model": {"type": "yolo", "name": "yolo11s", "pretrained": True},
#     "training": {"epochs": 20, "batch_size": 16, "optimizer": "SGD", "lr": 0.01},
# }
#
# # 데이터 재준비 (tier 변경 시)
# train_coco_s, val_coco_s = create_subset(coco, tier="small", seed=42)
# yaml_s = prepare_yolo_dataset(train_coco_s, val_coco_s, DATA_DIR / "yolo_small")
# run_yolo_experiment(custom_config, yaml_s, EXPERIMENTS_DIR)